# LAB 3:  
LÀM SẠCH DỮ LIỆU CƠ BẢN

Vấn đề 1: Tiến hành tải dữ liệu vào chương trình ứng dụng Python và giải quyết vấn đề
“Missing header in the csv file”

In [1]:
import pandas as pd

column_names = [
    "Id", "Name", "Age", "Weight",
    "m0006", "m0612", "m1218",
    "f0006", "f0612", "f1218"
]

df = pd.read_csv("patient_heart_rate.csv",names=column_names,usecols=range(10))
print(df.head())

FileNotFoundError: [Errno 2] No such file or directory: 'patient_heart_rate.csv'

Vấn đề 2: Xử lý vấn đề một cột lưu hỗn hợp nhiều dữ liệu, ở đây là cột “Name” chứa bao
gồm “Firstname” và “Lastname”, giải pháp là ta sẽ tách ra làm 2 cột  

In [ ]:
df[['Firstname','Lastname']] = df['Name'].str.split(expand=True)
df = df.drop('Name', axis=1)
print (df)

Vấn đề 3: Cột Weight có vấn đề về không thống nhất các đơn vị đo lường trong dữ liệu.
Ta sẽ chuyển các đơn vị về thành đơn vị chuẩn “kg”

In [ ]:
weight = df['Weight']

for i in range (0 ,len(weight)):
    x= str(weight[i])
    if "lbs" in x[-3:]:
        x = x[:-3]
        float_x = float(x)
        y =int(float_x/2.2)
        y = str(y)+"kgs"
        weight[i]= y
print (df)

Vấn đề 4: Vấn đề về xuất hiện dòng dữ liệu rỗng (không có giá trị: NaN). Giải pháp có
thể đưa ra là xóa bỏ

In [ ]:
df.dropna(how='all', inplace=True)
print(df)

Vấn đề 5: Có nhiều dòng dữ liệu bị trùng lắp thông tin hoàn toàn[fullname, lastname,
age, weight,....], giải pháp đưa ra là chỉ giữ lại một dòng dữ liệu, tuy nhiên giải pháp phải
dựa trên nghiệp vụ của tập dữ liệu và quan sát của người xử lý.

In [ ]:
df = df.drop_duplicates(subset=['Firstname','Lastname', 'Age', 'Weight'])
print (df)

Vấn đề 6: Xuất hiện dữ liệu bị ảnh hưởng bởi lỗi non-ASCII, không định dạng ASCII.
Giải pháp: Tùy vào nghiệp vụ ta có thể: xóa dữ liệu tại đó, thay thế bằng dữ liệu khác
hoặc thay bằng việc đánh dấu bằng một kí tự khác (ví dụ: ‘warning’)

In [ ]:
df.Firstname.replace({r'[^\x00-\x7F]+':''}, regex=True, inplace=True)
df.Lastname.replace({r'[^\x00-\x7F]+':''}, regex=True, inplace=True)
print (df)

Vấn đề 7



In [ ]:
# Thống kê số lượng dòng bị thiếu dữ liệu trên hai cột Age và Weight
print("Số lượng dữ liệu thiếu trong cột Age:")
print(df['Age'].isnull().sum())

print("\nSố lượng dữ liệu thiếu trong cột Weight:")
print(df['Weight'].isnull().sum())

In [ ]:
#Xử lý dữ liệu thiếu theo yêu cầu
df.dropna(subset=['Age', 'Weight'], how='all', inplace=True)
age_mean = df['Age'].mean()
df['Age'].fillna(age_mean, inplace=True)
print(df)

 Vấn đề 8: “một cột chứa quá nhiều thông tin cần được phân rã”, như trong bài toán này ta
thấy header “m0006” chứa các nội dung bao gồm: m → male, 1218 ~ 12-18 (mm-dd).
Còn giá trị thì là kết quả huyết áp.

In [ ]:
df = pd.melt(df, id_vars=['Id', 'Age', 'Weight', 'Firstname', 'Lastname'], value_name="PulseRate", var_name="sex_and_time").sort_values(['Id', 'Age', 'Weight', 'Firstname', 'Lastname'])

tmp_df = df['sex_and_time'].str.extract(r'(\D)(\d+)(\d{2})', expand=True)

tmp_df.columns = ["Sex", "hours_lower", "hours_upper"]

tmp_df["Time"] = tmp_df["hours_lower"] + "-" + tmp_df["hours_upper"]

df = pd.concat([df, tmp_df], axis=1)

df = df.drop(['sex_and_time', 'hours_lower', 'hours_upper'], axis=1)
df = df.dropna()
df.to_csv('outputcleanup.csv', index=False)
print (df)

Vấn đề 9

In [ ]:
df['PulseRate'] = pd.to_numeric(df['PulseRate'], errors='coerce')

missing_ratio = df['PulseRate'].isnull().mean() * 100
print(f"Tỉ lệ dữ liệu thiếu trên cột PulseRate: {missing_ratio:.2f}%")

df = df.sort_values(by=['Id', 'Time']).reset_index(drop=True)

prev_1 = df.groupby('Id')['PulseRate'].shift(1)
prev_2 = df.groupby('Id')['PulseRate'].shift(2)
next_1 = df.groupby('Id')['PulseRate'].shift(-1)
next_2 = df.groupby('Id')['PulseRate'].shift(-2)

method_1 = (prev_1 + next_1) / 2
df['PulseRate'].fillna(method_1, inplace=True)

method_2 = (prev_1 + prev_2) / 2
df['PulseRate'].fillna(method_2, inplace=True)

method_3 = (next_1 + next_2) / 2
df['PulseRate'].fillna(method_3, inplace=True)

method_4 = df.groupby('Id')['PulseRate'].transform('mean')
df['PulseRate'].fillna(method_4, inplace=True)

method_5 = df.groupby('Sex')['PulseRate'].transform('mean')
df['PulseRate'].fillna(method_5, inplace=True)

global_mean = df['PulseRate'].mean()
df['PulseRate'].fillna(global_mean, inplace=True)

print(df)

Hãy rút gọn dữ liệu phù hợp và reindex lại dữ liệu. Sau đó, lưu trữ dữ liệu đã xử lý thành
công với tên file patient_heart_rate_clean.csv

In [ ]:
cleaned_columns = ['Id', 'Firstname', 'Lastname', 'Age', 'Weight', 'Sex', 'Time', 'PulseRate']
df_clean = df[cleaned_columns]

df_clean.reset_index(drop=True, inplace=True)

df_clean.to_csv('patient_heart_rate_clean.csv', index=False)

print(df_clean.head(10))